# Lesson 23 Lab — Accuracy Regression Tests for Quantized Models

**Puzzle:** Can one aggregate score hide a serious quantization regression?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

Quality evidence spans token likelihood (cross-entropy/perplexity), task metrics, output/logit agreement, safety/alignment cases, and business-specific slices.

### Core mechanism

Perplexity is `exp(mean token cross-entropy)`; a small average loss change can coexist with large ranking changes on a rare slice. Top-1 agreement reveals decision changes but not whether either answer is correct.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "23-accuracy-regression"
device = require_cuda()
torch.manual_seed(2026 + 23)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

Large suites improve coverage but slow iteration. A tiered gate uses fast deterministic smoke/regression samples first and expensive benchmarks before release.

### What this code tests

The CUDA probe computes loss, perplexity, overall agreement, and two slices from identical hidden states before and after INT4 Q/DQ.

**Experiment:** Run a tiny CUDA language-model head before and after INT4 weight Q/DQ, then compare cross-entropy, perplexity, top-1 agreement, and slice metrics.

**Declared evidence label:** `pytorch-gpu`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
torch.manual_seed(7); vocab,hidden,tokens=256,128,4096; h=torch.randn(tokens,hidden,device=device); w=torch.randn(vocab,hidden,device=device); targets=torch.randint(0,vocab,(tokens,),device=device)
base=h@w.t(); _,_,dq=symmetric_quantize(w,bits=4,group_size=64); cand=h@dq.t()
base_loss=torch.nn.functional.cross_entropy(base,targets); cand_loss=torch.nn.functional.cross_entropy(cand,targets)
slices={"first_half":slice(0,tokens//2),"second_half":slice(tokens//2,None)}; slice_rows={}
for name,s in slices.items(): slice_rows[name]={"top1_agreement":round((base[s].argmax(-1)==cand[s].argmax(-1)).float().mean().item(),6)}
result=base_result(23,"pytorch-gpu"); result.update({"synthetic_probe":{"tokens":tokens,"vocab":vocab,"baseline_loss":round(base_loss.item(),7),
    "candidate_loss":round(cand_loss.item(),7),"baseline_perplexity":round(base_loss.exp().item(),5),"candidate_perplexity":round(cand_loss.exp().item(),5),
    "top1_agreement":round((base.argmax(-1)==cand.argmax(-1)).float().mean().item(),6),"slices":slice_rows},
    "conclusion":"Multiple frozen metrics exposed the synthetic INT4 regression; they are not scores for a named language model."})


## 3. Inspect the evidence

Apply predeclared gates to every metric and slice. This synthetic probe is not a benchmark score for a named LLM.

### Acceptance and rollback gate

Freeze datasets, prompts, decoding, baseline revision, thresholds, and slice definitions. Fail on a critical slice even if the global average passes.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "Multiple frozen metrics exposed the synthetic INT4 regression; they are not scores for a named language model.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:46:09+00:00",
  "lesson": 23,
  "schema_version": 1,
  "synthetic_probe": {
    "baseline_loss": 32.0494919,
    "baseline_perplexity": 82969305808896.0,
    "candidate_loss": 32.2126198,
    "candidate_perplexity": 97670416826368.0,
    "slices": {
      "first_half": {
        "top1_agreement": 0.838379
      },
      "second_half": {
        "top1_agreement": 0.835449
      }
    },
    "tokens": 4096,
    "top1_agreement": 0.836914,
    "vocab": 256
  }
}
Saved: artifacts/rtx5090-result.json


## 4. Explain the result

Use a layered quality gate and retain the baseline outputs needed to explain a regression.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).